# Phase 3 — Industry Revenue at Risk Analysis

This notebook estimates which Thai industries have the highest semiconductor-linked exposure using a Revenue-at-Risk model.

## Model

Revenue at Risk = Production Value × Semiconductor Cost Share × Taiwan Proxy Dependency

The Taiwan proxy dependency comes from Phase 2 and uses `Other Asia, nes` as a Taiwan/Chinese Taipei proxy with limitations.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

INPUT_PATH = Path("../data/processed/industry_revenue_at_risk_inputs.csv")
OUTPUT_PATH = Path("../data/processed/industry_revenue_at_risk_results.csv")
VISUALS_DIR = Path("../visuals")
VISUALS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(INPUT_PATH)
df

In [ ]:
for case in ["low", "base", "high"]:
    df[f"revenue_at_risk_{case}_usd_bn"] = (
        df["production_value_usd_bn"] *
        df[f"chip_share_{case}"] *
        (df["taiwan_dependency_pct"] / 100)
    )

def risk_level(x):
    if x >= 5:
        return "Very High"
    elif x >= 1:
        return "High"
    else:
        return "Moderate"

df["risk_level"] = df["revenue_at_risk_base_usd_bn"].apply(risk_level)
df = df.sort_values("revenue_at_risk_base_usd_bn", ascending=False).reset_index(drop=True)
df.insert(0, "rank", range(1, len(df) + 1))
df.to_csv(OUTPUT_PATH, index=False)
df

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=df["industry"],
    y=df["revenue_at_risk_base_usd_bn"],
    error_y=dict(
        type="data",
        symmetric=False,
        array=df["revenue_at_risk_high_usd_bn"] - df["revenue_at_risk_base_usd_bn"],
        arrayminus=df["revenue_at_risk_base_usd_bn"] - df["revenue_at_risk_low_usd_bn"]
    ),
    name="Base case"
))
fig.update_layout(
    title="Estimated Semiconductor Revenue at Risk by Thai Industry",
    xaxis_title="Industry",
    yaxis_title="Revenue at Risk (USD bn)",
    template="plotly_white"
)
fig.write_html(VISUALS_DIR / "industry_revenue_at_risk_bar.html")
fig.show(renderer="browser")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=df["industry"], y=df["production_value_usd_bn"], name="Production value"))
fig.add_trace(go.Scatter(x=df["industry"], y=df["revenue_at_risk_base_usd_bn"], mode="lines+markers", name="Revenue at risk", yaxis="y2"))
fig.update_layout(
    title="Production Value vs. Semiconductor Revenue at Risk",
    xaxis_title="Industry",
    yaxis=dict(title="Production Value (USD bn)"),
    yaxis2=dict(title="Revenue at Risk (USD bn)", overlaying="y", side="right"),
    template="plotly_white",
    legend=dict(orientation="h")
)
fig.write_html(VISUALS_DIR / "production_value_vs_risk.html")
fig.show(renderer="browser")